<a href="https://colab.research.google.com/github/changeroa/FineMining/blob/main/Multilanguage_Speech_recognition.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
!pip install -U openai-whisper
!sudo apt update && sudo apt install ffmpeg

Hit:1 https://cloud.r-project.org/bin/linux/ubuntu jammy-cran40/ InRelease
Hit:2 https://developer.download.nvidia.com/compute/cuda/repos/ubuntu2204/x86_64  InRelease
Get:3 http://security.ubuntu.com/ubuntu jammy-security InRelease [129 kB]
Hit:4 https://r2u.stat.illinois.edu/ubuntu jammy InRelease
Hit:5 http://archive.ubuntu.com/ubuntu jammy InRelease
Get:6 http://archive.ubuntu.com/ubuntu jammy-updates InRelease [128 kB]
Hit:7 https://ppa.launchpadcontent.net/deadsnakes/ppa/ubuntu jammy InRelease
Hit:8 https://ppa.launchpadcontent.net/graphics-drivers/ppa/ubuntu jammy InRelease
Hit:9 https://ppa.launchpadcontent.net/ubuntugis/ppa/ubuntu jammy InRelease
Get:10 http://archive.ubuntu.com/ubuntu jammy-backports InRelease [127 kB]
Get:11 http://archive.ubuntu.com/ubuntu jammy-updates/main amd64 Packages [2,696 kB]
Fetched 3,080 kB in 2s (1,640 kB/s)
Reading package lists... Done
Building dependency tree... Done
Reading state information... Done
52 packages can be upgraded. Run 'apt list -

In [ ]:
import whisper

# could it be you calling me down, 안녕하세요 라고 녹음했습니다.
model = whisper.load_model("large")
ko_result = model.transcribe("./sample.m4a", language="ko", no_speech_threshold=0.3, logprob_threshold=-2.0)
en_result = model.transcribe("./sample.m4a", language="en", no_speech_threshold=0.3, logprob_threshold=-2.0)
print(ko_result["text"])
print(en_result["text"])

/usr/local/lib/python3.10/dist-packages/whisper/__init__.py:150: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  checkpoint = torch.load(fp, map_location=device)
/usr/local/li

 그리피 유 콜링 미 다운. 안녕하세요.
 Creepy you calling me down. Annyeonghaseyo


In [ ]:
import torch
from transformers import AutoTokenizer, AutoModel
from sklearn.metrics.pairwise import cosine_similarity
import numpy as np

# 다국어 임베딩 모델 로드
model_name = "sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2"
tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModel.from_pretrained(model_name)

words_1 = ko_result["text"].split()
words_2 = en_result["text"].split()

# 어절 단위 임베딩 계산
def get_embeddings(text_list):
    inputs = tokenizer(text_list, padding=True, truncation=True, return_tensors="pt")
    with torch.no_grad():
        outputs = model(**inputs)
    # 임베딩 벡터 추출 (pooling)
    embeddings = outputs.last_hidden_state.mean(dim=1).numpy()
    return embeddings

embeddings_1 = get_embeddings(words_1)
embeddings_2 = get_embeddings(words_2)

# 어절 간 코사인 유사도 계산
similarities = cosine_similarity(embeddings_1, embeddings_2)

# 유사도가 높은 어절 선택
selected_words = []
for i, word in enumerate(words_1):
    if i < len(words_2) and similarities[i, i] > 0.5:  # 유사도 임계값
        selected_words.append(words_2[i])
    else:
        selected_words.append(word)

# 최종 텍스트 구성
final_text = " ".join(selected_words)
print(f"Final text: {final_text}")

/usr/local/lib/python3.10/dist-packages/transformers/tokenization_utils_base.py:1601: FutureWarning: `clean_up_tokenization_spaces` was not set. It will be set to `True` by default. This behavior will be depracted in transformers v4.45, and will be then set to `False` by default. For more details check this issue: https://github.com/huggingface/transformers/issues/31884
  warnings.warn(


Final text: Creepy you calling me down. 안녕하세요.
